# Demo 6 - Peer-group anomaly detection (UEBA) with clustering

**Fast** (per-user aggregate) · **Pool:** Medium · **Visual:** 2-D cluster scatter

**The question:** who is behaving unlike the people they normally resemble?

Rather than testing every user against one fixed threshold, this notebook lets the data
sort users into behaviour groups on its own, then flags the people sitting furthest from
the middle of their own group. An admin gets compared against other admins, not against
the sales team.

Nothing here needs labelled examples of "bad", so it works on a tenant nobody has tuned it
for. That combination - grouping entities by behaviour, then reducing several measurements
to a picture you can read - has no equivalent in KQL.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `WORKSPACE` / `LOOKBACK_DAYS` - which workspace, and how much history to profile.
- `N_CLUSTERS` - how many behaviour groups to sort users into. Four is a sensible start for
  a mixed workforce (roughly: desk workers, admins, automation-heavy accounts, travellers).
  There is no correct number. Change it and watch the picture change.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 14
N_CLUSTERS = 4

## 3. Build a behaviour profile for every user

We are not looking for bad events here. We are describing how each person *normally*
behaves, in five numbers:

| Feature | What it means |
| --- | --- |
| `total` | How many times they signed in |
| `fail_ratio` | What share of those attempts failed |
| `ips` | How many different source IP addresses they came from |
| `apps` | How many different applications they touched |
| `offhrs_ratio` | What share of their activity fell outside 07:00-19:00 |

Sign-ins with no `UserPrincipalName` are dropped. Those are app and service-principal
logins, and if you leave them in they all collapse into a single fake "user" that wins
every anomaly ranking and tells you nothing.

Users with fewer than five sign-ins are excluded too. There is not enough there to call it
a pattern.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pyspark.sql.types import StructType, StructField, StringType

df = data_provider.read_table("SigninLogs", WORKSPACE)
df = df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))

status = StructType([StructField("errorCode", StringType(), True)])
ok = ["0","50125","50140","70043","70044"]
df = (df.withColumn("ec", F.from_json("Status", status).getField("errorCode"))
        .withColumn("fail", (~F.col("ec").isin(ok)).cast("int"))
        .withColumn("offhrs", ((F.hour("TimeGenerated")<7)|(F.hour("TimeGenerated")>=19)).cast("int")))

# Without this, every UPN-less sign-in collapses into one phantom "user" that lands as a
# guaranteed outlier and tells the SOC nothing.
df = df.filter(F.col("UserPrincipalName").isNotNull() &
               (F.trim(F.col("UserPrincipalName")) != ""))

feat = (df.groupBy("UserPrincipalName").agg(
            F.count("*").alias("total"),
            F.avg("fail").alias("fail_ratio"),
            F.countDistinct("IPAddress").alias("ips"),
            F.countDistinct("AppDisplayName").alias("apps"),
            F.avg("offhrs").alias("offhrs_ratio"))
          .filter(F.col("total") >= 5)).toPandas()
print("users profiled:", len(feat))

## 4. Group users by behaviour, then measure who does not fit

Three pieces of standard data science. In plain terms:

**StandardScaler** puts the five features on a common scale. Without it, `total` (which
might be in the thousands) would completely drown out `fail_ratio` (which sits between 0
and 1), and the maths would only ever "see" how much somebody signs in.

**K-Means** sorts users into `N_CLUSTERS` groups so that people in the same group behave
similarly to each other. Nobody tells it what the groups are or what to look for - it finds
the groupings itself from the shape of the data. That is what *unsupervised* means, and it
is why this works on a tenant you have never seen before, with no tuning and no labelled
examples of "bad".

**Distance to your own group's centre** is the actual anomaly score. A user sitting far
from the middle of their own peer group is behaving unlike the people they otherwise most
resemble. That is a far better question than "did anyone cross a fixed threshold", because
the comparison is against their actual peers rather than against a number somebody guessed.

**PCA** (principal component analysis) then squashes those five features down to two
numbers, so the result can be drawn on a flat chart while keeping as much of the spread as
possible. The two axes, PC1 and PC2, are blends of the original five features and have no
real-world units. Ignore the numbers on the axes and read the shape.

If there are fewer users than clusters, the cell prints a message and skips rather than
crashing.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

cols = ["total","fail_ratio","ips","apps","offhrs_ratio"]

if len(feat) < max(N_CLUSTERS, 3):
    print(f"Only {len(feat)} users profiled - need at least {max(N_CLUSTERS, 3)} to cluster. "
          "Raise LOOKBACK_DAYS or lower the minimum sign-in count.")
    outliers = feat.head(0)
else:
    X = StandardScaler().fit_transform(feat[cols].fillna(0))

    # Fit once and reuse - labels and centres must come from the same model.
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42).fit(X)
    feat["cluster"] = km.labels_
    # Distance to own cluster centre = "how odd is this user"
    feat["dist"] = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1)
    xy = PCA(n_components=2, random_state=42).fit_transform(X)
    feat["x"], feat["y"] = xy[:,0], xy[:,1]
    outliers = feat.nlargest(8, "dist")

## 5. Plot the peer groups and star the outliers

Each dot is one user, coloured by which behaviour group they landed in. Dots near each
other represent people who behave alike.

The gold stars are the eight users furthest from their own group's centre, labelled with
their UPN. Those are the ones to look at first, and the table underneath gives you their
raw five features so you can see which one is doing the work.

**What to look for:** a star sitting *between* two clusters, or well outside all of them. A
star on the outer edge of a big tight cluster is usually just a busy person having a busy
fortnight.

In [ ]:
if "x" not in feat.columns:
    print("Clustering was skipped above - nothing to plot.")
else:
    plt.figure(figsize=(11, 7))
    plt.scatter(feat["x"], feat["y"], c=feat["cluster"], cmap="tab10", s=40, alpha=.7)
    plt.scatter(outliers["x"], outliers["y"], marker="*", s=380, edgecolor="black",
                facecolor="gold", label="top outliers", zorder=5)
    for _, r in outliers.iterrows():
        plt.annotate(str(r["UserPrincipalName"])[:22], (r["x"], r["y"]),
                     fontsize=8, xytext=(5,5), textcoords="offset points")
    plt.title("User behaviour peer groups (PCA) - outliers starred")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.tight_layout(); plt.show()

outliers[["UserPrincipalName"]+cols+["dist"]] if not outliers.empty else "no outliers"

## Why a notebook beats KQL here

K-Means, StandardScaler and PCA are core data-science, not query operators. KQL can't cluster entities or reduce five features to a 2-D map - so it can't answer *'who behaves unlike their peers?'* the way this does.